# PersistBench — Gate G7 populated-store sensitivity arm (Kaggle)

Runs `scripts/g7_arm.py` with `configs/g7_arm.json`: the Gate G2 smoke grid — Qwen2.5-7B-Instruct + Qwen3-8B x 2 chains (chain01 tracking, chain05 stock) x 2 faults (silent_wrong_value, stale_value) x 3 policies (M0/M1/M5), dependent mode revising, seed 0 = **24 chain runs, 72 dependent outcomes** — re-run with the memory store **pre-populated with true distractor entries** from other chains' completed tasks (amendment §F.15; design note `docs/g7-populated-store-design.md`).

**Reuse, don't regenerate.** The 24 origin trajectories + stored writes already exist in the pilot outputs (`outputs/pilot_writes.jsonl` for Qwen2.5; `results/qwen3-repilot/pilot_writes.jsonl` for Qwen3 — both tracked, so the clone below has them). The arm generates **only the 72 dependent answers live** — minutes on a T4, not a second pilot. If a store cannot be reused, the script writes `fallback_pilot_config.json` and exits 2; the fallback (full regeneration of the 24 origins + writes) is documented in `persistbench/g7_arm.py` and is NOT run here.

Gate G7 is a **sensitivity measurement, not an effect gate**: there is no PASS/FAIL. The readout is the populated-vs-empty comparison in `g7_arm_summary.md` — propagation / persistence / correction resistance on the G6-eligible subset, per model, against the recorded G2b empty-store numbers (loaded from `results/g2b-smoke/g2_smoke_summary.json`, also tracked). Sunil et al. predict attenuation from pre-existing legitimate memory; the arm does not gate on direction. **Exit 0 = the arm completed** (grid ran, outputs written); exit 2 = reuse failure.


## 1. GPU check — hard fail without CUDA

The smoke models are 7-8B at 4-bit; a CPU run would take hours and a silent Accelerator=None commit must never burn a session. This cell raises if no GPU is visible.


In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NO GPU: set Accelerator = GPU T4 x2 before Save & Run All, then re-run.'
print('device:', torch.cuda.get_device_name(0))

## 2. Get the code

Option A: public GitHub repo (set the URL). Option B: the repo uploaded into the working directory. The cell clones into `/kaggle/working/persistbench` or uses the uploaded copy, then verifies the G7 prep is present (the arm script, its config, the module, the two pilot writes files the arm reuses, and the recorded G2b summary the comparison reads).


In [ ]:
import os, subprocess

REPO_URL = os.environ.get('PERSISTBENCH_REPO_URL', 'https://github.com/karmendra8386/persistbench-bench.git')  # published Nov 1, 2026; env var overrides
DEST = '/kaggle/working/persistbench'
if os.path.isdir(os.path.join(DEST, 'persistbench')):
    print('using uploaded repo copy at', DEST)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, DEST], check=True)
    print('cloned', REPO_URL, '->', DEST)

need = ['scripts/g7_arm.py', 'configs/g7_arm.json', 'persistbench/g7_arm.py',
        'outputs/pilot_writes.jsonl', 'results/qwen3-repilot/pilot_writes.jsonl',
        'results/g2b-smoke/g2_smoke_summary.json']
missing = [f for f in need if not os.path.exists(os.path.join(DEST, f))]
assert not missing, ('G7 prep missing from the repo copy — commit + push it first, or upload it: ' + ', '.join(missing))
print('G7 prep present; pilot writes files found:')
for f in need[3:5]:
    print(' ', f, os.path.getsize(os.path.join(DEST, f)), 'bytes')
print(subprocess.run(['git', '-C', DEST, 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip(), '(repo HEAD)')


## 3. Install dependencies — preserving Kaggle's CUDA torch

Kaggle's image ships the CUDA torch build the smoke needs; a naive install can replace it. The packages below do not pin torch, and the cell re-checks CUDA afterwards. (transformers 5.x requires accelerate for device_map; bitsandbytes supplies the 4-bit quantisation.)


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'bitsandbytes'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA lost after install — stop here, do not run the smoke.'
print('deps installed; cuda still available')

## 4. Run the arm

24 chain runs (2 models x 2 chains x 2 faults x 3 policies), 72 dependent answers, seed 0, revising dependents. The M1/M5 stores are pre-populated with the distractor seeds (chain01: 4 entries, chain05: 7 entries); M0 cells are the unseeded baseline, identical to the smoke's. Expected wall time: minutes on a T4 x2 — the origin trajectories and stored writes are reused from the pilot outputs, so no formation GPU is spent. Results land in `/kaggle/working/g7-arm`.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/g7_arm.py',
     '--config', 'configs/g7_arm.json',
     '--out', '/kaggle/working/g7-arm'],
    cwd='/kaggle/working/persistbench',
)
print('arm exit code:', result.returncode)
print('exit 0 = arm COMPLETED (a measurement, not a verdict — read the comparison in the summary); exit 2 = reuse failure, fallback config written — do NOT re-run blindly, read the message first')


## 5. Outputs — retrieval list

Expected files in `/kaggle/working/g7-arm` (also the version Output tab): `g7_arm_summary.md` (distractor seeds + the populated-vs-empty comparison table), `g7_arm_summary.json`, `g7_arm_cells.csv`, `g7_arm_cells.jsonl`, and `transcripts/` (one JSON per dependent outcome: the seeded store the dependent actually saw, the unseeded pilot write, the distractor seed provenance, prompt, transcript, parse log, usage). On a reuse failure the directory instead holds `fallback_pilot_config.json` — retrieve it, do not re-run.


In [ ]:
import os

OUT = '/kaggle/working/g7-arm'
for name in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, name)
    size = os.path.getsize(path) if os.path.isfile(path) else len(os.listdir(path))
    print(f'{name:32} {size:>10} {"bytes" if os.path.isfile(path) else "files"}')

print()
print(open(os.path.join(OUT, 'g7_arm_summary.md')).read())


## After retrieval (not in this notebook)

1. Copy `/kaggle/working/g7-arm` into the repo as `results/g7-arm/` and commit it.
2. Write the G7 record (`docs/g7-populated-store-results.md`) from the summary: the distractor seeds used, answer yield, the populated-vs-empty comparison (propagation / persistence / correction resistance on the eligible subset, Wilson CIs, per model only) and what it does to the §F.15 question — attenuation, no change, or amplification at smoke scale, stated with the smoke's n=2-chains caveat — plus an amendment entry recording the arm and its readout.
3. File the record to Drive §03 + update the EB1 tracker in the same pass (repo rule).
4. G7 is the last of gates G1–G7. With it discharged, the empty-store scope limitation does not need to go into the paper's abstract; the scale-up decision itself is a separate go.
